# w26_gpu1_pinns_jrdb_ekr6_emergence
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `GPU T4 x2` | Internet: `On`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Kaggle Cloud environment required'
REV = 'f90f0122cda14a6768426a6e6987736e2e9fc062'
with tempfile.TemporaryDirectory(prefix='w26_g1_') as ts:
    tmp = Path(ts)
    repo, meta_dir, vcache = tmp / 'repo', tmp / 'pinns_meta', tmp / 'vcache'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env4 = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'OMP_NUM_THREADS': '4', 'MKL_NUM_THREADS': '4'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env4, check=True)
    out_zip = WORK / 'wave26_pinns_jrdb_ekr6_emergence.zip'
    subprocess.run([
        sys.executable, '-u', '-m', 'scripts.run_wave26_pinns_jrdb_ekr6_emergence',
        '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz),
        '--joint-zip', str(joint_bin), '--causal-map-bin', str(map_bin),
        '--repo-root', str(repo), '--video-cache-dir', str(vcache), '--zip-path', str(out_zip)
    ], cwd=repo, env=env4, check=True)
    print('DONE:', out_zip, out_zip.stat().st_size, hashlib.sha256(out_zip.read_bytes()).hexdigest(), flush=True)
